In [2]:
from gridstatus.ercot_api.ercot_api import ErcotAPI
from dotenv import load_dotenv
import polars as pl
from pathlib import Path

data_dir = Path("data")
inputs_data_dir = data_dir / "inputs" # inputs because this is an input to downstream analysis
inputs_data_dir.mkdir(exist_ok=True, parents=True)

load_dotenv()

start_date_cpt = "2025-10-01"
end_date_cpt = "2025-10-08"
chosen_node = "RHESS2_ESS1" # "ENDPARK_ESS1"

In [3]:
ercot_api = ErcotAPI()

In [4]:
ercot_api.list_all_public_endpoints()

/np1-301/60_cop_adj_period_snapshot
    60-Day COP Adjustment Period Snapshot
/np3-233-cd/hourly_res_outage_cap
    Hourly Resource Outage Capacity
/np3-565-cd/lf_by_model_weather_zone
    Seven-Day Load Forecast by Model and Weather Zone
/np3-566-cd/lf_by_model_study_area
    Seven-Day Load Forecast by Model and Study Area
/np3-765-cd/approved_dctie_schedules
    Approved DC Tie Schedules
/np3-910-er/2d_agg_dsr_loads
    2-Day Aggregated DSR Loads
/np3-910-er/2d_agg_gen_summary
    2-Day Aggregated Generation Summary
/np3-910-er/2d_agg_gen_summary_houston
    2-Day Aggregated Generation Summary Houston
/np3-910-er/2d_agg_gen_summary_north
    2-Day Aggregated Generation Summary North
/np3-910-er/2d_agg_gen_summary_south
    2-Day Aggregated Generation Summary South
/np3-910-er/2d_agg_gen_summary_west
    2-Day Aggregated Generation Summary West
/np3-910-er/2d_agg_load_summary
    2-Day Aggregated Load Summary
/np3-910-er/2d_agg_load_summary_houston
    2-Day Aggregated Load Summary Ho

In [5]:
ercot_api.describe_one_public_endpoint("/np6-788-cd/lmp_node_zone_hub")
# ercot_api.describe_one_public_endpoint("/np6-970-cd/rtd_lmp_node_zone_hub")

Endpoint: /np6-788-cd/lmp_node_zone_hub
Summary:  LMPs by Resource Nodes, Load Zones and Trading Hubs
Parameters:
    LMPFrom - float
    LMPTo - float
    Ocp-Apim-Subscription-Key - string
    SCEDTimestampFrom - timestamp
    SCEDTimestampTo - timestamp
    repeatHourFlag - boolean
    settlementPoint - string


In [6]:
actual_lmps_pd = ercot_api.get_lmp_by_settlement_point(
    date=start_date_cpt,
    end=end_date_cpt
)

2025-12-03 10:48:13 - INFO - Requesting url: https://api.ercot.com/api/public-reports/np6-788-cd/lmp_node_zone_hub with params: {'size': 100000, 'SCEDTimestampFrom': '2025-10-01T00:00:00', 'SCEDTimestampTo': '2025-10-08T00:00:00', 'page': 1}
2025-12-03 10:48:16 - INFO - Requesting url: https://api.ercot.com/api/public-reports/np6-788-cd/lmp_node_zone_hub with params {'size': 100000, 'SCEDTimestampFrom': '2025-10-01T00:00:00', 'SCEDTimestampTo': '2025-10-08T00:00:00', 'page': 2}
2025-12-03 10:48:16 - INFO - Requesting url: https://api.ercot.com/api/public-reports/np6-788-cd/lmp_node_zone_hub with params: {'size': 100000, 'SCEDTimestampFrom': '2025-10-01T00:00:00', 'SCEDTimestampTo': '2025-10-08T00:00:00', 'page': 2}
2025-12-03 10:48:18 - INFO - Requesting url: https://api.ercot.com/api/public-reports/np6-788-cd/lmp_node_zone_hub with params {'size': 100000, 'SCEDTimestampFrom': '2025-10-01T00:00:00', 'SCEDTimestampTo': '2025-10-08T00:00:00', 'page': 3}
2025-12-03 10:48:18 - INFO - Reque

In [7]:
indicative_lmps_pd = ercot_api.get_indicative_lmp_by_settlement_point(
    date=start_date_cpt,
    end=end_date_cpt
)

2025-12-03 10:49:22 - DEBUG - Getting historical data for np6-970-cd from 2025-10-01 00:00:00-05:00 to 2025-10-08 00:00:00-05:00
2025-12-03 10:49:22 - INFO - Requesting url: https://api.ercot.com/api/public-reports/archive/np6-970-cd with params: {'postDatetimeFrom': '2025-10-01T00:00:00', 'postDatetimeTo': '2025-10-08T00:00:00', 'size': 1000, 'page': 1}
2025-12-03 10:49:23 - INFO - Requesting url: https://api.ercot.com/api/public-reports/archive/np6-970-cd with params: {'postDatetimeFrom': '2025-10-01T00:00:00', 'postDatetimeTo': '2025-10-08T00:00:00', 'size': 1000, 'page': 2}
2025-12-03 10:49:24 - INFO - Requesting url: https://api.ercot.com/api/public-reports/archive/np6-970-cd with params: {'postDatetimeFrom': '2025-10-01T00:00:00', 'postDatetimeTo': '2025-10-08T00:00:00', 'size': 1000, 'page': 3}
2025-12-03 10:49:24 - INFO - Found 2016 archives
2025-12-03 10:49:24 - DEBUG - Bulk downloading historical data
2025-12-03 10:49:24 - INFO - Requesting url: https://api.ercot.com/api/publ

In [8]:
actual_lmps = pl.from_pandas(actual_lmps_pd)


In [9]:
indicative_lmps = pl.from_pandas(indicative_lmps_pd)

In [10]:
filt_actual_lmps = (
    actual_lmps
    .filter(pl.col('Location') == pl.lit(chosen_node))
    .rename({
        'SCED Timestamp': 'sced_timestamp',
        'Interval Start': 'interval_start_cpt',
        'Interval End': 'interval_end_cpt',
        'LMP': 'lmp',
    })
    .select('sced_timestamp', 'interval_start_cpt', 'interval_end_cpt', 'lmp')
    .unique(subset=['interval_end_cpt'], keep='first')
    # don't try to deal with multiple SCED dispatches in a given 5-minute interval
    .sort('sced_timestamp')
    .select('interval_start_cpt', 'interval_end_cpt', 'lmp')
)
filt_actual_lmps

interval_start_cpt,interval_end_cpt,lmp
"datetime[ns, US/Central]","datetime[ns, US/Central]",f64
2025-10-01 00:00:00 CDT,2025-10-01 00:05:00 CDT,22.67
2025-10-01 00:05:00 CDT,2025-10-01 00:10:00 CDT,23.38
2025-10-01 00:10:00 CDT,2025-10-01 00:15:00 CDT,22.98
2025-10-01 00:15:00 CDT,2025-10-01 00:20:00 CDT,25.18
2025-10-01 00:20:00 CDT,2025-10-01 00:25:00 CDT,24.18
…,…,…
2025-10-07 23:35:00 CDT,2025-10-07 23:40:00 CDT,33.46
2025-10-07 23:40:00 CDT,2025-10-07 23:45:00 CDT,33.24
2025-10-07 23:45:00 CDT,2025-10-07 23:50:00 CDT,32.35


In [11]:
filt_indicative_lmps = (
    indicative_lmps
    .filter(pl.col('Location') == pl.lit(chosen_node))
    .rename({
        'RTD Timestamp': 'sced_timestamp',
        'Interval Start': 'interval_start_cpt',
        'Interval End': 'interval_end_cpt',
        'LMP': 'lmp',
    })
    .select('sced_timestamp', 'interval_start_cpt', 'interval_end_cpt', 'lmp')
    .sort('sced_timestamp')
    # Floor sced_timestamp to 5-minute intervals
    .with_columns(
        pl.col('sced_timestamp').dt.truncate('5m').alias('as_of_cpt')
    )
    # Get the first (earliest) sced_timestamp per 5-minute window
    .with_columns(
        pl.col('sced_timestamp').min().over('as_of_cpt').alias('first_sced_timestamp')
    )
    # Keep only rows from the first SCED run in each 5-minute window (all ~12 forecasts)
    .filter(pl.col('sced_timestamp') == pl.col('first_sced_timestamp'))
    .drop('first_sced_timestamp')
    # Calculate horizon as number of 5-min intervals between as_of and interval_end
    .with_columns(
        horizon_5min_idx=(
            (pl.col("interval_end_cpt") - pl.col("as_of_cpt"))
            .dt.total_seconds()
            / (5 * 60)
        ).cast(pl.Int32)
    )
    .select('as_of_cpt', 'interval_start_cpt', 'interval_end_cpt', 'horizon_5min_idx', 'lmp')
)
filt_indicative_lmps

as_of_cpt,interval_start_cpt,interval_end_cpt,horizon_5min_idx,lmp
"datetime[ns, US/Central]","datetime[ns, US/Central]","datetime[ns, US/Central]",i32,f64
2025-10-01 00:00:00 CDT,2025-10-01 00:00:00 CDT,2025-10-01 00:05:00 CDT,1,22.67
2025-10-01 00:00:00 CDT,2025-10-01 00:05:00 CDT,2025-10-01 00:10:00 CDT,2,22.55
2025-10-01 00:00:00 CDT,2025-10-01 00:10:00 CDT,2025-10-01 00:15:00 CDT,3,21.93
2025-10-01 00:00:00 CDT,2025-10-01 00:15:00 CDT,2025-10-01 00:20:00 CDT,4,21.84
2025-10-01 00:00:00 CDT,2025-10-01 00:20:00 CDT,2025-10-01 00:25:00 CDT,5,21.79
…,…,…,…,…
2025-10-07 23:55:00 CDT,2025-10-08 00:25:00 CDT,2025-10-08 00:30:00 CDT,7,27.34
2025-10-07 23:55:00 CDT,2025-10-08 00:30:00 CDT,2025-10-08 00:35:00 CDT,8,26.8
2025-10-07 23:55:00 CDT,2025-10-08 00:35:00 CDT,2025-10-08 00:40:00 CDT,9,26.19


In [12]:
# Check that the set of as_of_utc's in indicative equals interval_start_cpt's in actual
indicative_as_of_series = filt_indicative_lmps['as_of_cpt'].unique()
actual_interval_start_series = filt_actual_lmps['interval_start_cpt'].unique()

print(f"Indicative as_of_cpt count: {len(indicative_as_of_series)}")
print(f"Actual interval_start_cpt count: {len(actual_interval_start_series)}")

# Use semi-join to filter to shared timestamps
# First, create lookup dataframes with matching column names
actual_lookup = filt_actual_lmps.select(pl.col('interval_start_cpt').alias('ts')).unique()
indicative_lookup = filt_indicative_lmps.select(pl.col('as_of_cpt').alias('ts')).unique()

# Filter actual to only timestamps that exist in indicative
inner_filt_actual_lmps = (
    filt_actual_lmps
    .join(indicative_lookup, left_on='interval_start_cpt', right_on='ts', how='semi')
)

# Filter indicative to only timestamps that exist in actual
inner_filt_indicative_lmps = (
    filt_indicative_lmps
    .join(actual_lookup, left_on='as_of_cpt', right_on='ts', how='semi')
)

print(f"\ninner_filt_actual_lmps rows: {len(inner_filt_actual_lmps)}")
print(f"inner_filt_indicative_lmps rows: {len(inner_filt_indicative_lmps)}")
print(f"inner_filt_actual_lmps unique timestamps: {inner_filt_actual_lmps['interval_start_cpt'].n_unique()}")
print(f"inner_filt_indicative_lmps unique as_of_cpt: {inner_filt_indicative_lmps['as_of_cpt'].n_unique()}")


Indicative as_of_cpt count: 2016
Actual interval_start_cpt count: 2011

inner_filt_actual_lmps rows: 2011
inner_filt_indicative_lmps rows: 22121
inner_filt_actual_lmps unique timestamps: 2011
inner_filt_indicative_lmps unique as_of_cpt: 2011


In [13]:
filename_actual_lmps = f"{start_date_cpt}_to_{end_date_cpt}_{chosen_node}_actual_lmps"
inner_filt_actual_lmps.write_csv(
    inputs_data_dir / f"{filename_actual_lmps}.csv",
    datetime_format="%Y-%m-%d %H:%M:%S"
)

In [14]:
filename_indicative_lmps = f"{start_date_cpt}_to_{end_date_cpt}_{chosen_node}_indicative_lmps"
inner_filt_indicative_lmps.write_csv(
    inputs_data_dir / f"{filename_indicative_lmps}.csv",
    datetime_format="%Y-%m-%d %H:%M:%S"
)